# Analysis and Plotting · turning `timings.csv` into figures

This is **labDD**, the fourth and final on-ramp. Its job is to teach you the pandas + matplotlib workflow that every measurement lab from lab 01 onward assumes you know.

It is **optional but strongly recommended** if any of these are true:

- You have never used pandas.
- You have plotted with matplotlib but never with a house style, log axes, or multi-series legends.
- You don't know what a **speedup** or **parallel efficiency** plot is.
- You have heard "roofline" and never made one.

If all four are already comfortable, skip straight to lab 00 — you'll lose nothing.

**Everything runs on this Hub.** No cluster round-trips. You will practice on a synthetic multi-lab `timings.csv` first, then when your real lab 01+ runs accumulate rows, the same code works.

**You will:**
1. Load a CSV with pandas; inspect shape, dtypes, first/last rows.
2. Filter and group — pick out one lab's runs, aggregate by thread count, sort.
3. Plot: line, bar, log-log, multi-series with legend.
4. Compute **speedup** and **parallel efficiency** from a `timings.csv` schema and plot them.
5. Draw your first **roofline plot** and place a data point on it.
6. Bridge to the rest of the course — the fixed CSV schema every lab appends to, and how `plotScaling` from `labHelpers.py` bundles the common patterns.

> **📚 Where to look when you're stuck**
> 
> - [**pandas User Guide**](https://pandas.pydata.org/docs/user_guide/index.html) — > the canonical reference. Read the "10 minutes to pandas" page first.
> - [**matplotlib gallery**](https://matplotlib.org/stable/gallery/index.html) — > paste a picture that looks like what you want, get the code that made it.
> - [**Roofline model on Wikipedia**](https://en.wikipedia.org/wiki/Roofline_model) — > the paper is [Williams, Waterman, Patterson 2009](https://people.eecs.berkeley.edu/~kubitron/cs252/handouts/papers/RooflineVyNoYellow.pdf).
> - [**LBNL's roofline toolkit**](https://cass.community/software/empirical-roofline-tool.html) > — the standard tool if you want to build your own roofline on Crux (lab 02 uses this).


## How this notebook works · Hub-only, one surface

| Where | How it looks | What it can do |
|---|---|---|
| **Hub** (this Jupyter kernel) | plain Python | pandas, matplotlib, numpy |


In [ ]:
# [Hub] Shared toolkit + the analysis libraries this lab uses.
from labHelpers import *
ensureDependencies(['pandas', 'matplotlib', 'numpy'])
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


### Set up this lab's identity


In [ ]:
# [Hub] Local-only lab; no cluster.
env = setupLab(labName="labDD", host="crux",
               remoteUser=os.environ.get("HPC_USER","student"),
               scratch="/tmp/labDDUnused")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] pandas + matplotlib installed (ensureDependencies did the work above).
preflight([
    check("pandas importable",     pythonImportable('pandas')),
    check("matplotlib importable", pythonImportable('matplotlib')),
    check("numpy importable",      pythonImportable('numpy')),
], infoRows=[('host','Hub (local)'), ('lab dir', str(labDir))])


## Part 1 · Load a CSV with pandas

The core object in pandas is a **DataFrame** — a 2-D table with named columns and typed values. Every measurement CSV in this course loads into a DataFrame in one line:

```python
df = pd.read_csv('timings.csv')
```

After that, `df.head()`, `df.shape`, `df.dtypes`, and `df.describe()` tell you what you just loaded before you write any real analysis code.


In [ ]:
# [Hub] Write a synthetic multi-lab timings.csv so you have realistic data
# BEFORE your own real runs accumulate. Same schema as lab 01's heat2D.c emits.
csvPath = labDir / 'timings.csv'
csvPath.write_text('lab,variant,N,steps,threads,ranks,wall_s,mlups,io_s\n01,serial,256,1000,1,1,3.24,20.2,0.31\n01,serial,512,1000,1,1,12.87,20.4,1.22\n01,serial,1024,1000,1,1,51.42,20.4,4.83\n03,openmp,256,1000,1,1,3.30,19.9,0.31\n03,openmp,256,1000,2,1,1.72,38.1,0.32\n03,openmp,256,1000,4,1,0.91,72.0,0.33\n03,openmp,256,1000,8,1,0.53,123.7,0.35\n03,openmp,256,1000,16,1,0.41,159.8,0.38\n03,openmp,256,1000,32,1,0.44,148.9,0.42\n06,mpi,1024,1000,1,1,51.98,20.2,4.85\n06,mpi,1024,1000,1,2,26.71,39.3,4.90\n06,mpi,1024,1000,1,4,13.98,75.1,4.99\n06,mpi,1024,1000,1,8,7.42,141.4,5.15\n06,mpi,1024,1000,1,16,4.28,245.2,5.37\n06,mpi,1024,1000,1,32,3.15,333.2,5.71\n07,hybrid,1024,1000,4,8,3.98,263.7,5.02\n07,hybrid,1024,1000,8,4,3.62,289.9,4.98\n07,hybrid,1024,1000,2,16,4.11,255.4,5.13\n')
showFile(csvPath, language='text', maxLines=10, title=f'timings.csv (first 10 lines of {sum(1 for _ in open(csvPath))-1} rows)')


In [ ]:
# [Hub] Load, inspect.
df = pd.read_csv(csvPath)
print('=== shape ===')
print(f'{df.shape[0]} rows x {df.shape[1]} columns')
print('\n=== dtypes ===')
print(df.dtypes)
print('\n=== head ===')
print(df.head())
print('\n=== describe (numeric summary) ===')
print(df[['N','threads','ranks','wall_s','mlups','io_s']].describe().round(2))


### 🖊️ What did you just see?

1. The `lab` column is currently loaded as `int64`. If you had labs `01A` and `01B` you'd want it as string — `pd.read_csv(..., dtype={'lab': str})` fixes that.
2. `describe()` shows min/max/mean/quartiles. Notice the wall_s range spans two orders of magnitude — some of these are cheap tests, some are big runs.
3. Every column name matches the schema `heat2D.c` writes. No mystery mapping.


In [ ]:
checkpoint("Part 1 - load and inspect a CSV", [
    check("timings.csv exists", fileExists(str(csvPath))),
    check("dataframe has expected columns",
          lambda: (set(df.columns) >= {'lab','variant','N','threads','ranks','wall_s','mlups','io_s'},
                   f"cols: {list(df.columns)}")),
])


## Part 2 · Filter, group, aggregate

Once you have a DataFrame, three operations do 90% of analysis:

- **Filter** — pick out rows: `df[df['variant'] == 'openmp']`
- **Sort** — order rows: `df.sort_values('threads')`
- **Group + aggregate** — collapse rows sharing a key: `df.groupby('N')['wall_s'].mean()`

These compose. `df[df.variant=='openmp'].groupby('threads')['wall_s'].mean()` gives you the mean wall time per thread count for just the OpenMP runs.


In [ ]:
# [Hub] Filter: just the OpenMP runs.
ompDf = df[df['variant'] == 'openmp'].sort_values('threads').reset_index(drop=True)
print('=== OpenMP runs (lab 03) ===')
print(ompDf.to_string(index=False))


In [ ]:
# [Hub] Group + aggregate: mean wall time per variant.
summary = df.groupby('variant').agg(
    runs=('wall_s', 'count'),
    minWall=('wall_s', 'min'),
    maxWall=('wall_s', 'max'),
    meanMlups=('mlups', 'mean'),
).round(2)
print('=== summary by variant ===')
print(summary)


In [ ]:
checkpoint("Part 2 - filter, group, aggregate", [
    check("OpenMP filter returned rows", lambda: (len(ompDf) > 0, f'{len(ompDf)} rows')),
    check("summary has 4 variants", lambda: (len(summary) == 4,
                                              f'{len(summary)} variants: {list(summary.index)}')),
])


## Part 3 · Plotting · line, bar, log-log, legend

Matplotlib figures always follow the same pattern:

```python
fig, ax = plt.subplots()      # create figure + one axes
ax.plot(xs, ys, label='name') # draw
ax.set_xlabel(...)            # decorate
ax.legend()
plt.show()                    # or fig.savefig('out.png')
```

Everything else is `ax.set_*` methods to control axes, ticks, titles, and grid lines.

Two things every HPC plot uses:

- **`ax.set_xscale('log')` / `set_yscale('log')`** — scaling plots span orders of magnitude; linear axes hide the interesting structure.
- **A legend with meaningful labels** — every series gets a `label=` in the `plot` call, then one call to `ax.legend()` at the end.


In [ ]:
# [Hub] Apply the course house style so every plot in this lab matches lab 01+.
palette = applyHouseStyle()


In [ ]:
# [Hub] Line plot: MLUP/s vs threads for OpenMP.
fig, ax = plt.subplots()
ax.plot(ompDf['threads'], ompDf['mlups'], marker='o', linestyle='-', label='OpenMP')
ax.set_xlabel('threads')
ax.set_ylabel('performance (MLUP/s)')
ax.set_xscale('log', base=2)
ax.set_title('OpenMP throughput vs thread count (N=256)')
ax.legend()
saveFigure(fig, 'ompThroughput', figuresDir=str(labDir/'figures'))


In [ ]:
# [Hub] Bar plot: wall time by variant for a matched problem (N=1024).
big = df[df['N'] == 1024].copy()
# For hybrid we have several ranks x threads combos; take the best (min wall).
bestPerVariant = big.groupby('variant')['wall_s'].min().sort_values()
fig, ax = plt.subplots()
ax.bar(bestPerVariant.index, bestPerVariant.values, color=palette[:len(bestPerVariant)])
ax.set_ylabel('best wall time (s)')
ax.set_title('N=1024 · best wall time by parallel variant')
saveFigure(fig, 'variantBestWall', figuresDir=str(labDir/'figures'))


In [ ]:
checkpoint("Part 3 - basic plots", [
    check("omp throughput plot exists", fileExists(str(labDir/'figures'/'ompThroughput.pdf'))),
    check("variant bar plot exists",    fileExists(str(labDir/'figures'/'variantBestWall.pdf'))),
])


## Part 4 · Speedup and parallel efficiency

When you parallelize, the two numbers you *always* report are **speedup** and **parallel efficiency**.

$$S(p) \;=\; \frac{T_{\text{serial}}}{T(p)} \qquad E(p) \;=\; \frac{S(p)}{p}$$

- **Speedup** $S(p)$: how many times faster with $p$ workers than serial. Ideal is $p$.
- **Efficiency** $E(p)$: fraction of ideal. 1.0 = perfect scaling, 0.5 = you're wasting half your cores.

On a **strong scaling** plot (fixed problem size), $S$ vs $p$ should track the ideal $y = x$ diagonal. When it deviates, either communication is eating into compute, or there's a serial fraction in the code that Amdahl's law forbids you to escape.

For a strict definition, see [Amdahl's law](https://en.wikipedia.org/wiki/Amdahl%27s_law) and [Gustafson's law](https://en.wikipedia.org/wiki/Gustafson%27s_law).


In [ ]:
# [Hub] Compute speedup + efficiency for the OpenMP runs.
# Serial baseline = the 1-thread row.
serialWall = ompDf.loc[ompDf['threads'] == 1, 'wall_s'].iloc[0]
ompDf['speedup']    = serialWall / ompDf['wall_s']
ompDf['efficiency'] = ompDf['speedup'] / ompDf['threads']
print(ompDf[['threads','wall_s','speedup','efficiency']].round(3).to_string(index=False))


In [ ]:
# [Hub] Twin-axis strong-scaling plot: speedup + efficiency in one figure.
fig, ax1 = plt.subplots()
ax1.plot(ompDf['threads'], ompDf['speedup'], marker='o', color=palette[0], label='measured speedup')
ax1.plot(ompDf['threads'], ompDf['threads'], linestyle=':', color='black', label='ideal (y=x)')
ax1.set_xlabel('threads')
ax1.set_ylabel('speedup', color=palette[0])
ax1.set_xscale('log', base=2)
ax1.set_yscale('log', base=2)
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(ompDf['threads'], ompDf['efficiency'], marker='s', color=palette[3], label='efficiency')
ax2.set_ylabel('efficiency', color=palette[3])
ax2.set_ylim(0, 1.1)
ax2.axhline(1.0, linestyle=':', color=palette[3], alpha=0.3)
ax2.legend(loc='upper right')

ax1.set_title('OpenMP strong scaling (N=256)')
saveFigure(fig, 'ompStrongScaling', figuresDir=str(labDir/'figures'))


### 🖊️ Read the plot

1. Where does speedup start deviating from ideal? (Look for the point at which the circles fall below the dotted diagonal.)
2. What is your efficiency at 32 threads? At 8?
3. If you had 64 or 128 cores, would you expect efficiency to climb, stay flat, or crash? Why? (Hint: think about the fixed problem size and Amdahl.)


In [ ]:
# [Hub] The same thing via labHelpers' plotScaling wrapper - one call, both axes.
# For lab work you'll probably use this helper rather than the manual twin-axis version above.
result = plotScaling(str(csvPath), kind='strong',
                     variantFilter='openmp',
                     baselineCol='threads', timeCol='wall_s',
                     outPath=str(labDir/'figures'/'ompStrongScalingHelper'))
print(f'plotScaling wrote: {result}')


In [ ]:
checkpoint("Part 4 - speedup + efficiency", [
    check("speedup computed",     lambda: ('speedup' in ompDf.columns, 'ok')),
    check("efficiency computed",  lambda: ('efficiency' in ompDf.columns, 'ok')),
    check("strong scaling plot exists", fileExists(str(labDir/'figures'/'ompStrongScaling.pdf'))),
])


## Part 5 · The roofline plot

[**The roofline model**](https://en.wikipedia.org/wiki/Roofline_model) is one plot that tells you whether your code is **compute-bound** or **memory-bound** on a given machine. You draw it once per machine; then every kernel you measure lands somewhere on it.

Axes:
- **x**: arithmetic intensity $I$ = (FLOPs performed) / (bytes moved from DRAM) — a property of your algorithm.
- **y**: performance $P$ in GFLOP/s — what you actually achieved.

Two ceilings on the y axis:
- **Peak compute**: the horizontal line at the machine's peak GFLOP/s.
- **Peak memory bandwidth**: the slanted line $P = I \cdot B_{\text{mem}}$ where $B_{\text{mem}}$ is the machine's peak DRAM bandwidth in GB/s.

The roofline is the *minimum* of the two — you can never go above either ceiling. Where they meet is the **ridge**. Kernels to the LEFT of the ridge are memory-bound; kernels to the RIGHT are compute-bound.

Lab 01's heat stencil is famously memory-bound: it does very little math per byte loaded, so it sits to the left of the ridge on essentially every machine.


In [ ]:
# [Hub] Build a synthetic roofline for a made-up 'small workstation'.
# Numbers here are for a plausible x86 desktop; substitute Crux's real values in
# lab 02, which measures them properly via the LBNL roofline toolkit.
peakCompute   = 800.0    # GFLOP/s (single-node peak)
peakBandwidth = 100.0    # GB/s   (DRAM peak)
ridge         = peakCompute / peakBandwidth   # arithmetic intensity at the corner

intensities = np.logspace(-2, 2, 200)   # x in FLOP/byte
perfCeiling = np.minimum(peakCompute, intensities * peakBandwidth)

fig, ax = plt.subplots()
ax.plot(intensities, perfCeiling, color='black', linewidth=2, label='roofline')
ax.axhline(peakCompute, linestyle=':', color=palette[0], alpha=0.5, label=f'peak compute {peakCompute:g} GFLOP/s')
ax.plot(intensities, intensities * peakBandwidth,
        linestyle=':', color=palette[3], alpha=0.5, label=f'peak BW {peakBandwidth:g} GB/s')
ax.axvline(ridge, linestyle=':', color='gray', alpha=0.4)
ax.text(ridge*1.15, peakCompute*0.15, f'ridge: I={ridge:.1f}', fontsize=9)

# Plot a synthetic heat-stencil measurement: I~0.2 FLOP/byte, P~15 GFLOP/s.
stencilI = 0.20
stencilP = 15.0
ax.plot([stencilI], [stencilP], marker='X', markersize=14, color=palette[1],
        label='2D heat stencil (measured)')

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('arithmetic intensity I (FLOP/byte)')
ax.set_ylabel('performance (GFLOP/s)')
ax.set_title('Roofline for a made-up workstation')
ax.set_xlim(0.01, 100); ax.set_ylim(1, 2000)
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='lower right', fontsize=8)
saveFigure(fig, 'roofline', figuresDir=str(labDir/'figures'))


### 🖊️ Read the roofline

1. The X marker (heat stencil) sits well below the slanted bandwidth ceiling. That means the code is **not** hitting peak DRAM bandwidth — there's room to improve by better reuse or blocking.
2. It also sits well to the left of the ridge, so no amount of extra compute (bigger vector units, more FMA throughput) would help — the bottleneck is bytes, not FLOPs.
3. **The horizontal distance from the X to the roofline** is how much of a memory-efficiency gap you have to close.

In lab 02 you'll build a real roofline for Crux — measured, not synthetic — using `likwid` or the [LBNL roofline toolkit](https://cass.community/software/empirical-roofline-tool.html) — and place your lab 01 heat-stencil measurement on it. Every subsequent parallel version of the stencil produces a new X on the same plot.


In [ ]:
checkpoint("Part 5 - roofline", [
    check("roofline plot exists", fileExists(str(labDir/'figures'/'roofline.pdf'))),
])


## Part 6 · Bridge to the rest of the course

You now have four artifacts in `~/labDD/figures/`: OpenMP throughput vs threads, variant comparison bar chart, strong-scaling speedup + efficiency, and a roofline. Every one of them was made from a **single CSV in the schema lab 01 defines**:

```
lab, variant, N, steps, threads, ranks, wall_s, mlups, io_s
```

This is the schema every lab from 01 onward appends to. Lab 03 (OpenMP) adds rows with `variant='openmp'` and varying `threads`. Lab 06 (MPI) adds rows with `variant='mpi'` and varying `ranks`. Lab 07 (hybrid) does both. By the end of the semester your one `timings.csv` has 50+ rows, and the code you wrote in Parts 3-5 of this lab produces the figures for your final report **without changes**.

### The `plotScaling` helper — one call, common shapes

`labHelpers.py` bundles the common patterns as `plotScaling(csvPath, kind=...)`:

| `kind=` | What it makes |
|---|---|
| `'strong'` | speedup + efficiency vs #workers (fixed problem size) |
| `'weak'` | efficiency vs #workers (problem scales with workers) |
| `'roofline'` | roofline background + your measurements as points |
| `'timeline'` | wall time vs date, for tracking regression over the semester |

Use `plotScaling` for the standard shape; drop back to raw matplotlib (Part 3) when you want something the helper doesn't cover.

### What you now know how to do

1. Load any lab's `timings.csv` with one line.
2. Filter, group, and aggregate to isolate the variant or problem size you care about.
3. Plot with the course house style (`applyHouseStyle`), save to PDF+PNG (`saveFigure`).
4. Compute speedup and efficiency and read what they say about scaling behavior.
5. Draw a roofline and know what a point on it means.

You don't need to memorize this. Bookmark; come back to Part N when Part N is what a lab asks you to do.


## Wrap up

On-ramp series complete: labAA (Linux + SSH), labBB (C toolchain), labCC (numerics), labDD (this one, analysis). Together they cover every prerequisite the course spine (lab 00 → lab 13) assumes. From lab 02 onward, this lab's four figures are the shapes every measurement lab produces.


### Lab scorecard


In [ ]:
labSummary("Analysis and Plotting")


---
### One-minute feedback

What worked, what didn't, what should be clearer. Anonymous to your classmates; goes straight to the instructor.


In [ ]:
feedback("Analysis and Plotting")
